In [1]:
import pandas as pd
import numpy as np
import heapq

In [3]:
df = pd.read_excel('dados/base_analise.xlsx')
sinan = pd.read_parquet('dados/2_df_recodificado.parquet')
sinan = sinan[sinan['ANO'].isin(range(2019,2025))]

## 1. Caracterização epidemiológica do escorpionismo
### Parte 1
1.1 Total de acidentes escorpiônicos no período de 2019–2024.  
1.2. Distribuição anual dos casos.  
1.3 Incidência média anual por 100.000 habitantes.  
1.4 Distribuição dos casos segundo faixas etárias de interesse.  
1.5 Número e proporção de casos MG segundo a definição operacional.  
1.6 Numero e proporção de MG ate 10 anos entre o total MG.  
1.7 Numero e proporção de MG ate 11A59 anos entre o total MG.  
1.8 Numero e proporção de MG 60+ entre o total MG.  
1.9 Numero de óbitos por ano de ocorrência.  
1.10 Numero de óbitos e prop. em relação ao total de casos.  
1.11 Num. e prop obitos 10 anos em relacao ao total obitos.  
1.12 Num. e prop obitos 11 a 59 anos em relacao ao total obitos.  
1.13 Num. e prop obitos 60+ em relacao ao total obitos.  
1.14 Tabela. Casos, obitos e proporções MG por região.

In [4]:
# Parte 1
print(
      '---------------------------------------------------------------------------- \n',
      '1.1 Total de acidentes entre 2019 e 2024:', df['TOTAL_CASOS'].sum(), '\n\n',

      
      '---------------------------------------------------------------------------- \n'
      '1.2 Distribuição anual dos casos:', sinan['ANO'].value_counts(), '\n\n',


      '---------------------------------------------------------------------------- \n'
      '1.3 Incidência média anual por 100 mil hab.:', df['INC_MEDIA_ANUAL'].describe(), '\n\n',


      '---------------------------------------------------------------------------- \n'
      '1.4 Distribuição dos casos por faixa etária:', '\n\n',
      'Total até 10 anos:', df['TOTAL_10A'].sum(), f"({df['TOTAL_10A'].sum() / df['TOTAL_CASOS'].sum()*100:.1f}%)", '\n',

      'Total de 11 a 59  anos:', df['TOTAL_11A59'].sum(), f"({df['TOTAL_11A59'].sum() / df['TOTAL_CASOS'].sum()*100:.1f}%)" '\n',

      'Total 60+:', df['TOTAL_60'].sum(),  f"({df['TOTAL_60'].sum() / df['TOTAL_CASOS'].sum()*100:.1f}%)" '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.5 Numero e proporção de casos MG:', df['MG'].sum(), f"({df['MG'].sum() / df['TOTAL_CASOS'].sum() * 100:.1f}%)", '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.6 Numero e proporção de MG ate 10 anos entre o total MG:', df['MG_10'].sum(), f"({df['MG_10'].sum()/df['MG'].sum()*100:.1f}%)", '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.7 Numero e proporção de MG ate 11A59 anos entre o total MG:', df['MG_11A59'].sum(), f"({df['MG_11A59'].sum()/df['MG'].sum()*100:.1f}%)", '\n\n',
      
      '---------------------------------------------------------------------------- \n',
      '1.8 Numero e proporção de MG 60+ entre o total MG:', df['MG_60'].sum(), f"({df['MG_60'].sum()/df['MG'].sum()*100:.1f}%)", '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.9 Numero de óbitos por ano de ocorrência:','\n', sinan[sinan['EVOLUCAO']=='Obito por ap'].groupby('ANO').size(), '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.10 Numero de óbitos e prop. em relação ao total de casos:', df['OBITOS'].sum(), f"({df['OBITOS'].sum()/df['TOTAL_CASOS'].sum()*100:.3f}%)", '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.11 Num. e prop obitos 10 anos em relacao ao total obitos:', df['OBITOS_10'].sum(), f"({df['OBITOS_10'].sum()/df['OBITOS'].sum()*100:.1f}%)", '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.12 Num. e prop obitos 11 a 59 anos em relacao ao total obitos:', df['OBITOS_11A59'].sum(), f"({df['OBITOS_11A59'].sum()/df['OBITOS'].sum()*100:.1f}%)", '\n\n',
      
      
      '---------------------------------------------------------------------------- \n',
      '1.13 Num. e prop obitos 60+ em relacao ao total obitos:', df['OBITOS_60'].sum(), f"({df['OBITOS_60'].sum()/df['OBITOS'].sum()*100:.1f}%)"

      )



---------------------------------------------------------------------------- 
 1.1 Total de acidentes entre 2019 e 2024: 239086 

 ---------------------------------------------------------------------------- 
1.2 Distribuição anual dos casos: ANO
2023    50125
2022    43619
2024    42148
2020    35275
2021    34437
2019    33484
Name: count, dtype: int64 

 ---------------------------------------------------------------------------- 
1.3 Incidência média anual por 100 mil hab.: count     645.000000
mean      295.713442
std       306.888537
min         0.000000
25%        50.795801
50%       192.607163
75%       445.729430
max      1590.079874
Name: INC_MEDIA_ANUAL, dtype: float64 

 ---------------------------------------------------------------------------- 
1.4 Distribuição dos casos por faixa etária: 

 Total até 10 anos: 22610 (9.5%) 
 Total de 11 a 59  anos: 166395 (69.6%)
 Total 60+: 50081 (20.9%)

 ---------------------------------------------------------------------------- 
 1.

In [5]:
# 1.14 Tabela. Casos, obitos e proporções MG por região 

import pandas as pd

# ============================================================
# TABELA EPIDEMIOLÓGICA POR REGIÃO
# ============================================================

# 1. Agregação regional
tabela_regiao = (
    df
    .groupby("REGIAO", as_index=False)
    .agg(
        POPULACAO=("POP_GERAL", "sum"),
        CASOS=("TOTAL_CASOS", "sum"),
        MG_10=("MG_10", "sum"),
        MG=("MG", "sum"),
        OBITOS=("OBITOS", "sum")
    )
)

# 2. Linha total do Estado
total_estado = pd.DataFrame({
    "REGIAO": ["Estado de São Paulo"],
    "POPULACAO": [df["POP_GERAL"].sum()],
    "CASOS": [df["TOTAL_CASOS"].sum()],
    "MG_10": [df["MG_10"].sum()],
    "MG": [df["MG"].sum()],
    "OBITOS": [df["OBITOS"].sum()]
})

# 3. Acrescentar o total às regiões e calcular os indicadores
tabela_regiao = (
    pd.concat(
        [tabela_regiao, total_estado],
        ignore_index=True
    )
    .assign(
        INC_MEDIA_ANUAL=lambda x:
            x["CASOS"] / (x["POPULACAO"] * 6) * 100000,

        ATE_MG_10_N_PCT=lambda x:
            x["MG_10"].astype(str) + " (" +
            (x["MG_10"] / x["MG"] * 100)
            .round(1).astype(str) + "%)",

        MG_N_PCT=lambda x:
            x["MG"].astype(str) + " (" +
            (x["MG"] / x["CASOS"] * 100)
            .round(1).astype(str) + "%)",

        OBITOS_N_LET=lambda x:
            x["OBITOS"].astype(str) + " (" +
            (x["OBITOS"] / x["CASOS"] * 100)
            .round(3).astype(str) + "%)"
    )
    [[
        "REGIAO",
        "POPULACAO",
        "CASOS",
        "INC_MEDIA_ANUAL",
        "ATE_MG_10_N_PCT",
        "MG_N_PCT",
        "OBITOS_N_LET"
    ]]
    .round({
        "INC_MEDIA_ANUAL": 1
    })
)

tabela_regiao.to_excel(r'resultados/tabela_regiao_epidemio.xlsx')

# tabela_regiao


## 2. Caracterização da rede PESA
2.1 Número e proporção de muni com PESA.  
2.2 Número e proporção de muni sem PESA
2.3 Proporção de muni com PESA Escorpionismo.  
2.4 Descritivo: número de Pesa por município com acesso local.  
2.5 Nomes dos Municípios com mais de 1 PESA
2.6 Distribuição acidentes entre municípios com e sem PESA  
2.7 Distribuição acidentes MG entre municípios com e sem PESA.  
2.8 Proporção de MG em municípios com e sem Pesa.  
2.9 Proporção de casos em municípios com e sem Pesa.  


In [10]:
# Parte 1 - Caracterização da rede PESA
print(
    '---------------------------------------------------------------------------- \n',
    f"2.1 Número e proporção de muni com PESA: {df[df['ACESSO_LOCAL']==1].shape[0]} ({round((df[df['ACESSO_LOCAL']==1].shape[0] / 645 * 100), 1)}%)", 
    '\n\n',
    
    '---------------------------------------------------------------------------- \n',
    f"2.2 Número e proporção de muni sem PESA: {df[df['ACESSO_LOCAL']==0].shape[0]} ({round((df[df['ACESSO_LOCAL']==0].shape[0] / 645 * 100), 1)}%)",
    '\n\n',

    '---------------------------------------------------------------------------- \n',
    f"2.3 Proporção de muni com PESA Escorp: {df[df['ACESSO_LOCAL']==1].shape[0]} ({round((df[df['ACESSO_LOCAL']==1].shape[0] / 242 * 100), 1)}%)",
    '\n\n',

    '---------------------------------------------------------------------------- \n',
    f"2.4 Descritivo: número de Pesa por município com acesso local: {df.loc[df['ACESSO_LOCAL']==1,['N_PESA']].describe().round(1)}",
    '\n\n',

    f"2.5 Municípios com mais de 1 PESA: {df.loc[(df['ACESSO_LOCAL']==1) & (df['N_PESA']>1),['MUNI','N_PESA']].sort_values('N_PESA', ascending=False)}",
    '\n\n',

    '---------------------------------------------------------------------------- \n',
    f"2.6 Distribuição acidentes entre municípios com e sem PESA: \n {df.groupby('ACESSO_LOCAL')['TOTAL_CASOS'].describe().T}",
    '\n\n',

    '---------------------------------------------------------------------------- \n',
    f"2.7 Distribuição acidentes MG entre municípios com e sem PESA: \n {df.groupby('ACESSO_LOCAL')['MG'].describe().T}",
    '\n\n',

    '---------------------------------------------------------------------------- \n',
    '2.8 Proporção de casos em municípios com e sem Pesa:', '\n',
     'Muni com Pesa:', (df.loc[df['ACESSO_LOCAL']==1,'TOTAL_CASOS'].sum() / df['TOTAL_CASOS'].sum() * 100).round(1),'\n',
     'Muni sem Pesa:', (df.loc[df['ACESSO_LOCAL']==0,'TOTAL_CASOS'].sum() / df['TOTAL_CASOS'].sum() * 100).round(1),
     '\n\n',

    
    '---------------------------------------------------------------------------- \n',    
    '2.9 Proporção de MG em municípios com e sem Pesa:', '\n',
    'MG em Muni com Pesa:', (df.loc[df['ACESSO_LOCAL']==1,'MG'].sum() / df['MG'].sum() * 100).round(1),'\n',
    'MG em Muni sem Pesa:', (df.loc[df['ACESSO_LOCAL']==0,'MG'].sum() / df['MG'].sum() * 100).round(1)
)




---------------------------------------------------------------------------- 
 2.1 Número e proporção de muni com PESA: 220 (34.1%) 

 ---------------------------------------------------------------------------- 
 2.2 Número e proporção de muni sem PESA: 425 (65.9%) 

 ---------------------------------------------------------------------------- 
 2.3 Proporção de muni com PESA Escorp: 220 (90.9%) 

 ---------------------------------------------------------------------------- 
 2.4 Descritivo: número de Pesa por município com acesso local:        N_PESA
count   220.0
mean      1.1
std       0.6
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       7.0 

 2.5 Municípios com mais de 1 PESA:                       MUNI  N_PESA
611              SAO PAULO       7
635  SAO JOSE DO RIO PRETO       5
619    SAO JOSE DOS CAMPOS       4
612              GUARULHOS       2
614               CAMPINAS       2
613                  BAURU       2
616                MARILIA       2
615        

In [ ]:
# Tabela 2. Total de casos entre municípios com e sem PESA
describe_total_mg = df.groupby('ACESSO_LOCAL')['MG'].describe().round(1).T
describe_total_casos = df.groupby('ACESSO_LOCAL')['TOTAL_CASOS'].describe().round(1).T

pd.concat([describe_total_mg, describe_total_casos], axis=1).to_excel('resultados/describe_casos_por_tipo_acesso.xlsx')

In [ ]:
# População por acesso local
print(df.groupby('ACESSO_LOCAL')['POP_GERAL'].sum())

ACESSO_LOCAL
0    10974105
1    34762873
Name: POP_GERAL, dtype: int64


In [ ]:
# Taxa de indicência acumulada por tipo de acesso
print(
    'Tx Incidencia em municipios SEM acesso local:', (df.loc[df['ACESSO_LOCAL']==0, 'TOTAL_CASOS'].sum()/df.loc[df['ACESSO_LOCAL']==0, 'POP_GERAL'].sum()*100000).round(1), '\n',
    'Tx Incidencia em municipios COM acesso local:', (df.loc[df['ACESSO_LOCAL']==1, 'TOTAL_CASOS'].sum()/df.loc[df['ACESSO_LOCAL']==1, 'POP_GERAL'].sum()*100000).round(1)
    )

Tx Incidencia em municipios SEM acesso local: 640.5 
 Tx Incidencia em municipios COM acesso local: 485.6


# 3. Acessibilidade geográfica aos PESA
* População de análise: municípios sem PESA local.
* Número de municípios dependentes de deslocamento intermunicipal.
* Tempo estimado até o PESA de referência:
    * mediana;
    * intervalo interquartil;
    * mínimo e máximo.
* Distribuição dos municípios segundo estratos de tempo.
* Distribuição regional dos tempos de deslocamento.
* População residente em cada estrato de acessibilidade.
* Número de acidentes e MG em cada estrato de tempo.
* Identificação dos municípios/regiões com maiores tempos estimados.
* Produto principal: Mapa do tempo estimado de deslocamento até o PESA de referência.


## 4. Sobreposição entre carga epidemiológica e acessibilidade
* População de análise: municípios sem PESA local.
* Construção de duas variáveis categóricas:
    * carga acumulada de MG entre 2019–2024: menor, intermediária e maior;
    * tempo estimado até o PESA: menor, intermediário e maior.
* Construção da matriz bivariada 3 × 3, resultando em nove perfis territoriais.
* Para cada perfil:
    * número de municípios;
    * população;
    * total de acidentes;
    * número de MG;
    * participação na carga de MG.
* Identificação dos municípios classificados simultaneamente como maior carga de MG + maior tempo de deslocamento.
* Avaliação estatística da sobreposição:
* qui-quadrado de Pearson;
* V de Cramér;
* frequências observadas e esperadas;
* resíduos padronizados.
* Interpretação como sobreposição territorial entre carga e acessibilidade, e não como efeito do tempo sobre a gravidade.
* Produto principal: Mapa bivariado carga de MG × tempo de acesso, acompanhado da matriz 3 × 3.
